In [3]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

In [4]:

INPUT_FILE = "fakenews.csv"
OUTPUT_FILE = "fakenews_preprocessed.csv"

df = pd.read_csv(INPUT_FILE)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


Dataset loaded successfully!
Dataset shape: (2212, 5)


In [5]:


def preprocess_text(text):

    # Handle missing values
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Convert text to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove email addresses
    text = re.sub(
        r"\b[\w.+-]+@[\w.-]+\.[a-z]{2,}\b",
        " ",
        text
    )

    # Preserve word boundaries when removing apostrophes
    # Example: don't -> dont
    text = re.sub(r"(?<=\w)['’](?=\w)", "", text)

    # Remove numbers
    text = re.sub(r"\d+", " ", text)

    # Remove punctuation, special characters,
    # and non-alphanumeric symbols.
    # Keep alphabetic characters and whitespace.
    text = "".join(
        char if char.isalpha() or char.isspace() else " "
        for char in text
    )

    # Remove extra spaces, tabs, and newlines
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [6]:
text_columns = ["headline", "body_text"]

for column in text_columns:

    if column not in df.columns:
        print(f"Warning: Column '{column}' not found.")
        continue

    df[f"{column}_preprocessed"] = (
        df[column].apply(preprocess_text)
    )

In [7]:
for column in text_columns:

    processed_column = f"{column}_preprocessed"

    if processed_column not in df.columns:
        continue

    # Check for digits
    has_digits = df[processed_column].str.contains(
        r"\d", regex=True, na=False
    ).any()

    # Check for punctuation or special characters
    has_special_chars = df[processed_column].str.contains(
        r"[^a-zA-Z\s]", regex=True, na=False
    ).any()

    print(f"\nValidation for {processed_column}:")
    print("Contains digits:", has_digits)
    print("Contains special characters:", has_special_chars)



Validation for headline_preprocessed:
Contains digits: False
Contains special characters: False

Validation for body_text_preprocessed:
Contains digits: False
Contains special characters: False


In [8]:
df.to_csv(OUTPUT_FILE, index=False)

print("\nPreprocessing completed successfully!")
print("Output file:", OUTPUT_FILE)
print("Final dataset shape:", df.shape)
print("\nPreprocessed dataset saved successfully!")


Preprocessing completed successfully!
Output file: fakenews_preprocessed.csv
Final dataset shape: (2212, 7)

Preprocessed dataset saved successfully!
